# 19 — Join Optimization Pattern

Purpose: reduce data before join to improve performance.

Core idea:

```text
GOOD:   filter → join
BAD:    join → filter
```

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("join-optimization-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions","8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input datasets

In [2]:
events = spark.createDataFrame(
    [(f"e{i}", f"u{i%5}", float(i)) for i in range(100)],
    ["event_id","user_id","amount"]
)

users = spark.createDataFrame(
    [(f"u{i}", f"country{i%3}") for i in range(5)],
    ["user_id","country"]
)

events.show(5)
users.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e0|     u0|   0.0|
|      e1|     u1|   1.0|
|      e2|     u2|   2.0|
|      e3|     u3|   3.0|
|      e4|     u4|   4.0|
+--------+-------+------+
only showing top 5 rows
+-------+--------+
|user_id| country|
+-------+--------+
|     u0|country0|
|     u1|country1|
|     u2|country2|
|     u3|country0|
|     u4|country1|
+-------+--------+



## Step 2 — BAD pattern (join then filter)

In [3]:
bad = (
    events
    .join(users, "user_id")
    .filter(F.col("amount") > 50)
)

bad.explain()
bad.count()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#1, event_id#0, amount#2, country#4]
   +- SortMergeJoin [user_id#1], [user_id#3], Inner
      :- Sort [user_id#1 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(user_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=49]
      :     +- Filter ((isnotnull(amount#2) AND (amount#2 > 50.0)) AND isnotnull(user_id#1))
      :        +- Scan ExistingRDD[event_id#0,user_id#1,amount#2]
      +- Sort [user_id#3 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(user_id#3, 8), ENSURE_REQUIREMENTS, [plan_id=50]
            +- Filter isnotnull(user_id#3)
               +- Scan ExistingRDD[user_id#3,country#4]




49

## Step 3 — GOOD pattern (filter before join)

In [4]:
good = (
    events
    .filter(F.col("amount") > 50)
    .join(users, "user_id")
)

good.explain()
good.count()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#1, event_id#0, amount#2, country#4]
   +- SortMergeJoin [user_id#1], [user_id#3], Inner
      :- Sort [user_id#1 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(user_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=285]
      :     +- Filter ((isnotnull(amount#2) AND (amount#2 > 50.0)) AND isnotnull(user_id#1))
      :        +- Scan ExistingRDD[event_id#0,user_id#1,amount#2]
      +- Sort [user_id#3 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(user_id#3, 8), ENSURE_REQUIREMENTS, [plan_id=286]
            +- Filter isnotnull(user_id#3)
               +- Scan ExistingRDD[user_id#3,country#4]




49

## Step 4 — Compare row counts

In [5]:
print("bad rows:", bad.count())
print("good rows:", good.count())

bad rows: 49
good rows: 49


## Step 5 — Why it matters

```text
filter before join = less shuffle data
join first = unnecessary data movement
```

In [6]:
spark.stop()